# Dataset Composition (Figure D1)

Replaces the draft's Figures 1 and 2 - nine sub-panels over two full pages - with one
five-panel figure. Panels a-d are the metadata breakdowns; panel e is the question/answer
composition that Figure 2 spent three bar charts on.

Built from `pathopen_data/processed/data.json`, not the raw CSV: the processed records
already normalise magnification (the raw sheet has `5X` and `5x` as separate values) and
express it as the combined microscope x eyepiece figure the paper quotes.


In [ ]:
import os
import sys
import json
import collections
import itertools

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np

import nature_style as ns

ns.apply_style()

DATA = os.path.join("..", "pathopen_data", "processed", "data.json")
with open(DATA) as handle:
    cases = json.load(handle)

print(f"cases: {len(cases)}")

BAR = ns.DATASET_COLORS["PathOPEN"]

# One colour per metadata field, shared by the Lorenz panel and reused wherever a field
# is named. Matches the field palette in the Pillar 5 figure so a reader tracking, say,
# Categorization sees the same hue in Data Records and in Technical Validation.
FIELD_COLORS = {
    "Categorization": ns.OKABE_ITO["vermillion"],
    "Organ": ns.OKABE_ITO["blue"],
    "Magnification": ns.OKABE_ITO["orange"],
    "Regional Anatomy": ns.OKABE_ITO["green"],
}


def cross_tab():
    """Cases per (organ system, broad category).

    Both fields are collapsed to their top level - "Gastrointestinal - Colon" to
    "Gastrointestinal", "Neoplasia (Malignant)" to "Neoplasia" - because the full
    43 x 32 grid is 96% empty and unreadable at panel size."""
    counts = collections.Counter()
    for case in cases:
        organ = case.get("Organ", "").split(" - ")[0].strip()
        category = case.get("Categorization", "").split(" (")[0].strip()
        if organ and category:
            counts[(organ, category)] += 1
    return counts


def magnification_matrix(mag_order):
    """Diagonal: cases holding ONLY that magnification.
    Off-diagonal: cases holding both. The two are different quantities - see caption."""
    position = {m: i for i, m in enumerate(mag_order)}
    matrix = np.full((len(mag_order), len(mag_order)), np.nan)
    pairs, solo = collections.Counter(), collections.Counter()
    for case in cases:
        mags = sorted({image["Magnification"] for image in case["Images"]
                       if image.get("Magnification")},
                      key=lambda m: float(str(m).lower().rstrip("x")))
        if len(mags) == 1:
            solo[mags[0]] += 1
        for first, second in itertools.combinations(mags, 2):
            pairs[(first, second)] += 1
    for (first, second), value in pairs.items():
        matrix[position[first], position[second]] = value
    for mag, value in solo.items():
        matrix[position[mag], position[mag]] = value
    return matrix


def gini(values):
    """Gini coefficient of a class-count vector; 0 is perfectly even."""
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    return (2.0 * np.sum(np.arange(1, n + 1) * v) / (n * v.sum())) - (n + 1) / n


# Standard pathology abbreviations for the organ-system prefix, plus a few long
# sub-organ and category names. Labels ran to 35 characters ("Gastrointestinal - Small
# Intenstine"), which forced a gutter wide enough to squeeze every panel in the row;
# shortening is the fix, not a narrower margin. The full strings stay in the data and in
# Appendix B - this only affects what is printed on the axis.
LABEL_ABBREVIATIONS = {
    "Gastrointestinal": "GI",
    "Musculoskeletal": "MSK",
    "Hematolymphoid": "Heme",
    "Central Nervous System": "CNS",
    "Hepatobiliary": "Hepatobil.",
    "Cardiovascular": "CV",
    "Reproductive": "Reprod.",
    "Respiratory": "Resp.",
}
LONG_TERMS = {
    "Small Intenstine": "Small Intest.",      # sic: spelling as collected
    "(Tissue degeneration)": "(degen.)",
    "(autoimmune)": "(AI)",
    "(Overgrowth)": "(overgr.)",
    "Intestinal villous mucosa": "Intestinal villous muc.",
    "Subcutaneous tissues": "Subcutaneous tiss.",
    "Squamous epithelium": "Squamous epith.",
}


def shorten(label):
    """Axis-label form of a metadata value; the stored value is unchanged."""
    text = label
    for full, short in LABEL_ABBREVIATIONS.items():
        if text.startswith(full + " - "):
            text = short + " - " + text[len(full) + 3:]
            break
    for full, short in LONG_TERMS.items():
        text = text.replace(full, short)
    return text


def top_counts(field, limit=None):
    """Counts for a case-level metadata field, most frequent first.

    Regional Anatomy is free text, so the same entry appears with different casing
    ("Mucosa" / "mucosa"). Fold case before counting, then display the most common
    spelling, or the tail inflates and a real class is split in two."""
    raw = [case[field].strip() for case in cases if case.get(field, "").strip()]
    by_key = collections.defaultdict(list)
    for value in raw:
        by_key[value.lower()].append(value)
    counts = collections.Counter()
    for key, spellings in by_key.items():
        counts[collections.Counter(spellings).most_common(1)[0][0]] = len(spellings)
    ordered = counts.most_common()
    if limit is not None and len(ordered) > limit:
        tail = sum(count for _, count in ordered[limit:])
        ordered = ordered[:limit] + [(f"other ({len(counts) - limit} classes)", tail)]
    return ordered, len(counts)


def hbar(ax, pairs, total_classes, xlabel, wrap=26):
    """Horizontal bars, most frequent at the top, count printed at the end of each."""
    labels = [shorten(label)[:wrap] for label, _ in pairs][::-1]
    values = [value for _, value in pairs][::-1]
    y = np.arange(len(values))
    colours = [BAR] * len(values)
    if pairs and pairs[-1][0].startswith("other"):
        colours[0] = "#BFBFBF"          # the pooled tail is not a class
    ax.barh(y, values, 0.72, color=colours)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=ns.fs(6.6))
    ax.set_xlabel(xlabel)
    for index, value in enumerate(values):
        ax.text(value + max(values) * 0.015, index, str(value), va="center",
                fontsize=ns.fs(6.6), color="#444444")
    ax.set_xlim(0, max(values) * 1.14)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return total_classes


In [ ]:
# --- panel e source: question/answer composition -------------------------------------
# The draft's Figure 2 used three separate bar charts (overall, single-image,
# multi-image) to carry six numbers. One stacked pair carries the same information and
# makes the split that actually matters visible: how many open-ended questions ship with
# a wrong answer as well as a correct one.
single, multi = collections.Counter(), collections.Counter()
for case in cases:
    bucket = single if len(case["Images"]) == 1 else multi
    for item in case["Open Ended"]:
        if not item.get("Question", "").strip():
            continue
        has_wrong = bool(item.get("Wrong Answer", "").strip())
        bucket["correct + wrong" if has_wrong else "correct only"] += 1

n_single = sum(1 for c in cases if len(c["Images"]) == 1)
n_multi = len(cases) - n_single
print(f"single-image cases {n_single}, multi-image {n_multi}")
print("single:", dict(single), " multi:", dict(multi))
print("open-ended questions:", sum(single.values()) + sum(multi.values()))


In [ ]:
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 232))

# One gridspec per row, each with its own left margin, rather than three rows nested in
# a single outer spec. Shared margins forced every row to the gutter panel a needs for
# "Gastrointestinal - Colon"; rows 2 and 3 start with much shorter labels and were
# paying for width they do not use.
# Row 1: the three case-level fields with few enough classes to show whole.
row1 = fig.add_gridspec(1, 3, width_ratios=[1.06, 1.00, 0.56], wspace=0.82,
                        left=0.148, right=0.980, top=0.968, bottom=0.752)

ax = fig.add_subplot(row1[0, 0])
pairs, total = top_counts("Organ", limit=11)
hbar(ax, pairs, total, "Cases")
ax.set_title(f"Organ system ({total} classes)", fontsize=7, pad=3)
ns.panel_label_below(ax, "a")

ax = fig.add_subplot(row1[0, 1])
pairs, total = top_counts("Categorization", limit=9)
hbar(ax, pairs, total, "Cases")
ax.set_title(f"Categorization ({total} classes)", fontsize=7, pad=3)
ns.panel_label_below(ax, "b")

ax = fig.add_subplot(row1[0, 2])
# Magnification is ordinal, so it keeps its natural order rather than being sorted by
# frequency like the nominal fields beside it.
mag_counts = collections.Counter(
    image["Magnification"] for case in cases for image in case["Images"]
    if image.get("Magnification"))
order = sorted(mag_counts, key=lambda m: float(str(m).lower().rstrip("x")))
y = np.arange(len(order))
ax.barh(y, [mag_counts[m] for m in order], 0.72, color=BAR)
ax.set_yticks(y)
ax.set_yticklabels(order, fontsize=ns.fs(7.2))
ax.invert_yaxis()
ax.set_xlabel("Images")
ax.set_title("Magnification", fontsize=7, pad=3)
for index, mag in enumerate(order):
    ax.text(mag_counts[mag] + 1.5, index, str(mag_counts[mag]), va="center",
            fontsize=ns.fs(6.6), color="#444444")
ax.set_xlim(0, max(mag_counts.values()) * 1.16)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ns.panel_label_below(ax, "c")

# Row 2: regional anatomy (long tail, so top 15) and the QA composition.
row2 = fig.add_gridspec(1, 3, width_ratios=[1.16, 0.74, 0.92], wspace=0.50,
                        left=0.150, right=0.980, top=0.648, bottom=0.426)

ax = fig.add_subplot(row2[0, 0])
# No pooled "other" bar here: the tail holds 100+ classes and would plot at ~112 against
# real classes of 2-6, flattening everything worth seeing. The remainder is stated in
# the title instead.
SHOW = 14
all_pairs, total = top_counts("Regional Anatomy")
shown = all_pairs[:SHOW]
remainder = sum(count for _, count in all_pairs[SHOW:])
hbar(ax, shown, total, "Cases", wrap=26)
ax.set_title(f"Regional anatomy (top {SHOW} of {total};\n{remainder} cases in the "
             f"remaining {total - SHOW} classes)", fontsize=ns.fs(6.8), pad=3,
             linespacing=1.3)
ns.panel_label_below(ax, "d")

ax = fig.add_subplot(row2[0, 1])
groups = [f"Single-image\n({n_single} cases)", f"Multi-image\n({n_multi} cases)"]
x = np.arange(len(groups))
correct_only = [single["correct only"], multi["correct only"]]
both = [single["correct + wrong"], multi["correct + wrong"]]
ax.bar(x, both, 0.52, color=BAR, label="correct + wrong answer")
ax.bar(x, correct_only, 0.52, bottom=both, color="#BFBFBF",
       label="correct answer only")
for index in range(len(groups)):
    if both[index]:
        ax.text(index, both[index] / 2, str(both[index]), ha="center", va="center",
                fontsize=ns.fs(6.8), color="white")
    if correct_only[index]:
        ax.text(index, both[index] + correct_only[index] / 2, str(correct_only[index]),
                ha="center", va="center", fontsize=ns.fs(6.8), color="#333333")
    total_q = both[index] + correct_only[index]
    ax.text(index, total_q + 6, str(total_q), ha="center", fontsize=ns.fs(6.8), color="#333333")
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=ns.fs(7))
ax.set_ylabel("Open-ended questions")
ax.set_ylim(0, max(b + c for b, c in zip(both, correct_only)) * 1.22)
ax.legend(loc="lower center", fontsize=ns.fs(7), frameon=False, ncol=1,
          handletextpad=0.45, labelspacing=0.28, borderaxespad=0.0,
          bbox_to_anchor=(0.5, 1.02))
ax.set_ylim(0, 300)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ns.panel_label_below(ax, "e")


# --- g: concentration of all four metadata fields -------------------------------------
# Panels a-d are necessarily truncated - 43 organ and 111 regional-anatomy classes do not
# fit as bars. This panel carries every class of every field with nothing pooled, and
# answers the question the bar charts cannot: how concentrated is each field? The
# diagonal is perfect balance; the further a curve falls below it, the more the cases
# pile into few classes.
#
# Gini here matches the values the Pillar 5 augmentation analysis reports, so the
# imbalance a reader meets in Data Records is the same number that gets corrected later.
row3 = fig.add_gridspec(1, 2, width_ratios=[0.84, 1.22], wspace=0.46,
                        left=0.062, right=0.980, top=0.330, bottom=0.100)
ax = fig.add_subplot(row3[0, 0])
ax.plot([0, 100], [0, 100], ls=":", lw=0.7, color="#999999", zorder=1)

LORENZ_FIELDS = ["Categorization", "Organ", "Magnification", "Regional Anatomy"]
for field in LORENZ_FIELDS:
    if field == "Magnification":
        # Magnification is counted per image, like panel c, because a case carries up to
        # three images at different magnifications. Sorting by frequency discards the
        # ordinal order, which is deliberate: the question here is concentration, not
        # which magnification is commonest - panel c already shows that.
        counts = np.array([mag_counts[m] for m in order], dtype=float)
        n_classes = len(counts)
    else:
        pairs, n_classes = top_counts(field)
        counts = np.array([count for _, count in pairs], dtype=float)
    ordered = np.sort(counts)
    cumulative = np.concatenate([[0.0], np.cumsum(ordered) / ordered.sum() * 100.0])
    shares = np.linspace(0, 100, len(cumulative))
    ax.plot(shares, cumulative, lw=1.3, color=FIELD_COLORS[field], zorder=3,
            label=f"{field} ({n_classes}, G={gini(counts):.2f})")

ax.set_xlabel("Cumulative share of classes (%)")
ax.set_ylabel("Cumulative share of cases (%)")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(fontsize=ns.fs(7), frameon=False, loc="upper left", handletextpad=0.5,
          labelspacing=0.3, borderaxespad=0.2, title="Field (classes, Gini)",
          title_fontsize=ns.fs(6.8))
ns.panel_label_below(ax, "g")


# --- h: organ system x category, the two-way coverage no bar panel shows ---------------
# Panels a-d give each field's marginal distribution; this gives their joint. The point
# is the sparsity: a reuser planning to stratify needs to know which combinations exist
# at all, not only which organs and which categories are common.
ax = fig.add_subplot(row3[0, 1])
counts = cross_tab()
organ_order = sorted({o for o, _ in counts},
                     key=lambda o: -sum(v for (a, _), v in counts.items() if a == o))
category_order = sorted({k for _, k in counts},
                        key=lambda k: -sum(v for (_, b), v in counts.items() if b == k))
N_ORGAN, N_CATEGORY = 13, 8
grid = np.array([[counts.get((o, k), 0) for k in category_order[:N_CATEGORY]]
                 for o in organ_order[:N_ORGAN]], dtype=float)
full = np.array([[counts.get((o, k), 0) for k in category_order]
                 for o in organ_order], dtype=float)

cmap = plt.cm.YlOrBr.copy()
cmap.set_bad("#F2F2F2")          # absent, not a low count
image = ax.imshow(np.ma.masked_where(grid == 0, grid), cmap=cmap, aspect="auto",
                  vmin=0, vmax=grid.max())
for row in range(grid.shape[0]):
    for column in range(grid.shape[1]):
        if grid[row, column] > 0:
            ax.text(column, row, f"{grid[row, column]:.0f}", ha="center", va="center",
                    fontsize=ns.fs(6.5),
                    color="white" if grid[row, column] > grid.max() * 0.6 else "#1f1f1f")
ax.set_xticks(range(grid.shape[1]))
ax.set_xticklabels(category_order[:N_CATEGORY], fontsize=ns.fs(6.8), rotation=90,
                   ha="center", va="top")
ax.set_yticks(range(grid.shape[0]))
ax.set_yticklabels([o.replace("Central Nervous System", "CNS")
                    for o in organ_order[:N_ORGAN]], fontsize=ns.fs(6.8))
ax.set_xticks(np.arange(-.5, grid.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, grid.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", lw=0.7)
ax.tick_params(which="minor", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
populated = 100.0 * np.count_nonzero(full) / full.size
shown_share = 100.0 * grid.sum() / full.sum()
ax.set_title(f"Organ system $\\times$ category (top {N_ORGAN}, {shown_share:.0f}% of cases)"
             f"\n{np.count_nonzero(full)} of {full.size} combinations populated "
             f"({populated:.0f}%)", fontsize=7, pad=4, linespacing=1.25)
ns.panel_label_below(ax, "h")

# --- f: which magnifications were captured together ------------------------------------
# Diagonal and off-diagonal carry DIFFERENT quantities on one scale; the caption says so
# explicitly and the diagonal is outlined. Kept because the pattern is the finding:
# multi-resolution coverage is local (adjacent steps), not spanning.
ax = fig.add_subplot(row2[0, 2])
matrix = magnification_matrix(order)
cmap = plt.cm.PuBu.copy()
cmap.set_bad("#F2F2F2")
image = ax.imshow(np.ma.masked_invalid(matrix), cmap=cmap, aspect="equal",
                  vmin=0, vmax=np.nanmax(matrix))
for row in range(len(order)):
    for column in range(len(order)):
        if not np.isnan(matrix[row, column]):
            ax.text(column, row, f"{matrix[row, column]:.0f}", ha="center", va="center",
                    fontsize=ns.fs(6.5),
                    color="white" if matrix[row, column] > np.nanmax(matrix) * 0.55
                    else "#1f1f1f",
                    fontweight="bold" if row == column else "normal")
    ax.add_patch(plt.Rectangle((row - 0.5, row - 0.5), 1, 1, fill=False,
                               edgecolor="#B33A3A", lw=0.9, zorder=4))
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, fontsize=ns.fs(6.8), rotation=90, ha="center", va="top")
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=ns.fs(6.8))
ax.set_xticks(np.arange(-.5, len(order), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(order), 1), minor=True)
ax.grid(which="minor", color="white", lw=0.7)
ax.tick_params(which="minor", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("Magnifications per case\n(outlined: that one only)", fontsize=7, pad=4,
             linespacing=1.25)
ns.panel_label_below(ax, "f")

paths = ns.save(fig, "figD1_dataset_composition")
print("wrote:", paths)
plt.show()


## Draft caption

**Fig. D1 | Composition of the PathOPEN dataset.**
**a**-**d**, Case-level metadata. Organ system (**a**), categorization (**b**) and
regional anatomy (**d**) are nominal and ordered by frequency; in **a** and **b** the grey
bar pools the classes below the cut, and **d** states its remainder in the title rather
than plotting it, because a 110-case bar would flatten classes holding two to eight cases.
Magnification (**c**) is ordinal, so it keeps its natural order, and is counted per image
rather than per case because a case may carry up to three images; values are the combined
microscope-objective x eyepiece figure.
**e**, Open-ended question composition. Every question carries a pathologist-authored
correct answer; the dark segment counts those that additionally carry a
pathologist-authored *wrong* answer, the feature no comparator dataset provides. All 74
questions from multi-image cases carry both.
**f**, Which magnifications were captured for the same case. **The diagonal and the
off-diagonal are different quantities.** An outlined diagonal cell counts cases holding
*only* that magnification (44 cases are 200x alone); an off-diagonal cell counts cases
holding *both* (24 cases pair 100x with 200x). They share one colour scale, which the
diagonal dominates, so read the panel for its pattern rather than for cross-cell
comparison. That pattern is the point: pairing is local, between adjacent steps
(40x-100x, 100x-200x), and the extremes never co-occur - no case holds both 20x and 400x.
Multi-resolution coverage therefore spans a narrow band, not the full range.
**g**, Concentration of all four metadata fields, every class included and nothing pooled.
Each curve gives the cumulative share of cases held by the cumulative share of classes,
ordered rarest to commonest; the dotted diagonal is perfect balance and the Gini
coefficient *G* measures the gap from it. The fields differ in kind, not only in size:
categorization is concentrated (*G* = 0.66, three classes hold 58% of cases) while
regional anatomy is dispersed (*G* = 0.24, 76% of its classes hold a single case). These
are the coefficients the augmentation analysis reports, so the imbalance shown here is
what the class-balanced variant corrects.
**h**, The joint distribution the marginal panels cannot show, with both fields collapsed
to their top level, showing the 13 largest organ systems against the 8 largest categories
(86% of cases). Only 62 of 399 organ-by-category combinations are populated (16%):
neoplasia appears in 18 of the 21 organ systems, while every other category is patchy. A reuser
planning to stratify needs the joint coverage, not the two marginals.

*Free-text metadata is matched case-insensitively before counting, merging entries such
as "Mucosa" and "mucosa", so class totals are slightly lower than a count of distinct
strings. Panels **f** and **h** show the densest rows and columns; the counts quoted in
their titles are over the full grids. Axis labels use the short forms listed in
Table S7; the released metadata carries the full values.*
